# SC4002: Natural Language Processing Project, Logistic Regression (Stage 1)

## Task: Build a system that identifies user's intent and extracts the slots in an English Assistance request. It involves two stages:
1. Task 1: Develop a baseline independently, without Generative AI assistance.
2. Task 2: Collaborate with AI to diagnose weaknesses, implement improvements.

Evaluate the resulting models from the above two stages. Submit reproducible notebook evidence, chat history with AI and a report of no more than two pages.

* Data can be found here "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz", en-US.jsonl file

In [1]:
# Necessary Files needed for the Project
import re
import time
import random
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Read json file
import json

# Data Manipulation Libraries
import numpy as np
import pandas as pd
import sklearn

# Visualisation Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

In [2]:
# Initialise all global variables
df = train_df = test_df = dev_df = pd.DataFrame()
label_encoder = LabelEncoder()

# Training Variables
number_of_epoch = 12

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python will run the model on:", device)
print("PyTorch version:", torch.__version__)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Python will run the model on: cpu
PyTorch version: 2.15.0


In [3]:
# Step 1: Extracting data from jsonl file to dataframes

try: 
    df = pd.read_json('data/en-US.jsonl', lines=True)
    print(df.head())
    print(df.columns)
except FileNotFoundError:
    print("ERROR: File not found")
except json.JSONDecodeError:
    print("ERROR: Invalid Json format in the file")

   id locale partition scenario             intent  \
0   0  en-US      test    alarm          alarm_set   
1   1  en-US     train    alarm          alarm_set   
2   2  en-US     train    alarm          alarm_set   
3   3  en-US      test    audio  audio_volume_mute   
4   4  en-US     train    audio  audio_volume_mute   

                                   utt  \
0      wake me up at five am this week   
1      wake me up at nine am on friday   
2  set an alarm for two hours from now   
3                                quiet   
4                           olly quiet   

                                           annot_utt  worker_id  
0  wake me up at [time : five am] [date : this week]          1  
1  wake me up at [time : nine am] on [date : friday]          1  
2       set an alarm for [time : two hours from now]          1  
3                                              quiet          1  
4                                         olly quiet          1  
Index(['id', 'locale', 'pa

## Data Cleaning

1. Drop columns such as 'locale' and 'worker_id'. These columns are unnecessary to the project.
2. Identify number of 'train', 'test' and 'dev' values in the DataFrame
3. Identify unique 'intent' and 'scenario'
4. Identify 'intent' and 'scenario' counts
5. Insert a new column 'intent_

In [4]:
# Step 1 in Data Cleaning
df = df.drop(columns=['locale', 'worker_id'])
print(df.columns)

Index(['id', 'partition', 'scenario', 'intent', 'utt', 'annot_utt'], dtype='str')


In [5]:
# Step 2 in Data Cleaning
# Print out how many train, test and dev datas in the Dataframe
print(df['partition'].value_counts())

partition
train    11514
test      2974
dev       2033
Name: count, dtype: int64


In [6]:
# Step 3 & 4 in Data Cleaning

# Identifying all values in 'intent' & count
print(df['intent'].unique())
print(df['intent'].value_counts())

<StringArray>
[               'alarm_set',        'audio_volume_mute',
      'iot_hue_lightchange',         'iot_hue_lightoff',
          'iot_hue_lighton',         'iot_hue_lightdim',
             'iot_cleaning',           'calendar_query',
               'play_music',           'general_quirky',
            'general_greet',           'datetime_query',
         'datetime_convert',           'takeaway_query',
             'alarm_remove',              'alarm_query',
               'news_query',           'music_likeness',
              'music_query',          'iot_hue_lightup',
           'takeaway_order',            'weather_query',
           'music_settings',        'audio_volume_down',
             'general_joke',        'music_dislikeness',
       'audio_volume_other',               'iot_coffee',
          'audio_volume_up',              'iot_wemo_on',
             'iot_wemo_off',                 'qa_stock',
               'play_radio',              'social_post',
 'recommendation_

In [7]:
# Step 3 & 4 in Data Cleaning (2)

# Identifying all values in 'scenario' & count
print(df['scenario'].unique())
print(df['scenario'].value_counts())

<StringArray>
[         'alarm',          'audio',            'iot',       'calendar',
           'play',        'general',       'datetime',       'takeaway',
           'news',          'music',        'weather',             'qa',
         'social', 'recommendation',        'cooking',      'transport',
          'email',          'lists']
Length: 18, dtype: str
scenario
calendar          2370
play              2024
qa                1685
email             1381
iot               1107
general            963
weather            855
transport          805
lists              793
news               709
recommendation     596
datetime           578
social             565
alarm              550
music              469
audio              387
takeaway           358
cooking            326
Name: count, dtype: int64


## Model Training (Logistic Regression)
Step by step
1. Reorganise the whole dataset into 3 dataframes (Training, Test, Dev)
2. Parse the slots per utterance into tokens and BIO tags
3. Verify annotations before training

In [8]:
# Step 1 in Model Training
train_df = df[df['partition'] == 'train'].copy()
test_df = df[df['partition'] == 'test'].copy()
dev_df = df[df['partition'] == 'dev'].copy()

slot_annotation_pattern = re.compile(r"\[([^:\]]+)\s*:\s*([^\]]+)\]")
word_pattern = re.compile(r"\S+")

# Step 2 in Model Training
def parse_annotated_utterance(annotated_utterance):
    original_text_parts = []
    slot_spans = []
    current_character_position = 0
    previous_match_end = 0

    for annotation_match in slot_annotation_pattern.finditer(annotated_utterance):
        text_before_slot = annotated_utterance[previous_match_end:annotation_match.start()]
        original_text_parts.append(text_before_slot)
        current_character_position += len(text_before_slot)

        slot_type = annotation_match.group(1).strip()
        slot_text = annotation_match.group(2)
        slot_start = current_character_position
        slot_end = slot_start + len(slot_text)

        original_text_parts.append(slot_text)
        slot_spans.append((slot_type, slot_start, slot_end))
        current_character_position = slot_end
        previous_match_end = annotation_match.end()

    original_text_parts.append(annotated_utterance[previous_match_end:])
    original_text = "".join(original_text_parts)
    return original_text, slot_spans


def tokenize_with_offsets(utterance):
    token_offsets = []
    for token_match in word_pattern.finditer(utterance):
        token_offsets.append((token_match.group(), token_match.start(), token_match.end()))
    return token_offsets


def convert_spans_to_bio(token_offsets, slot_spans):
    bio_labels = ["O"] * len(token_offsets)

    for slot_type, slot_start, slot_end in slot_spans:
        matching_token_indices = []
        for token_index, (token, token_start, token_end) in enumerate(token_offsets):
            if token_start < slot_end and token_end > slot_start:
                matching_token_indices.append(token_index)

        if not matching_token_indices:
            raise ValueError(f"No token found for slot {slot_type} at {slot_start}:{slot_end}")

        first_token_index = matching_token_indices[0]
        last_token_index = matching_token_indices[-1]
        first_token_start = token_offsets[first_token_index][1]
        last_token_end = token_offsets[last_token_index][2]

        if first_token_start != slot_start or last_token_end != slot_end:
            raise ValueError(f"Slot is not aligned with token boundaries: {slot_type}")

        if any(bio_labels[token_index] != "O" for token_index in matching_token_indices):
            raise ValueError("Overlapping slot annotations are not supported")

        bio_labels[first_token_index] = f"B-{slot_type}"
        for token_index in matching_token_indices[1:]:
            bio_labels[token_index] = f"I-{slot_type}"

    return bio_labels


sample_data = train_df.iloc[0]
sample_text, sample_slot_spans = parse_annotated_utterance(sample_data["annot_utt"])
sample_tokens = tokenize_with_offsets(sample_text)
sample_bio_labels = convert_spans_to_bio(sample_tokens, sample_slot_spans)

print("Original sentence:", sample_text)
print("Tokens:", [token for token, start, end in sample_tokens])
print("BIO labels:", sample_bio_labels)
print("Slot spans:", [(slot_type, sample_text[start:end]) for slot_type, start, end in sample_slot_spans])

Original sentence: wake me up at nine am on friday
Tokens: ['wake', 'me', 'up', 'at', 'nine', 'am', 'on', 'friday']
BIO labels: ['O', 'O', 'O', 'O', 'B-time', 'I-time', 'O', 'B-date']
Slot spans: [('time', 'nine am'), ('date', 'friday')]


In [12]:
# Step 3 in Machine Learning

# Training DataFrame
for i in range(len(train_df)):
    reconstructed_text, slots_spans = parse_annotated_utterance(train_df.iloc[i]["annot_utt"])
    assert reconstructed_text == train_df.iloc[i]["utt"], f"Text mismatch: Training Data, Data id: {train_df.iloc[i]["id"]}"        
    token_offsets = tokenize_with_offsets(train_df.iloc[i]['utt'])
    convert_spans_to_bio(token_offsets, slots_spans)

print("Training Dataframe successfully reconstructed and aligned successfully")

# Test DataFrame
for i in range(len(test_df)):
    reconstructed_text, slots_spans = parse_annotated_utterance(test_df.iloc[i]["annot_utt"])
    assert reconstructed_text == test_df.iloc[i]["utt"], f"Text mismatch: Test Data, Data id: {test_df.iloc[i]["id"]}"        
    token_offsets = tokenize_with_offsets(test_df.iloc[i]['utt'])
    convert_spans_to_bio(token_offsets, slots_spans)

print("Test Dataframe successfully reconstructed and aligned successfully")

# Dev DataFrame
for i in range(len(dev_df)):
    reconstructed_text, slots_spans = parse_annotated_utterance(dev_df.iloc[i]["annot_utt"])
    assert reconstructed_text == dev_df.iloc[i]["utt"], f"Text mismatch: Dev Data, Data id: {dev_df.iloc[i]["id"]}"        
    token_offsets = tokenize_with_offsets(dev_df.iloc[i]['utt'])
    convert_spans_to_bio(token_offsets, slots_spans)

print("Dev Dataframe successfully reconstructed and aligned successfully")

Training Dataframe successfully reconstructed and aligned successfully
Test Dataframe successfully reconstructed and aligned successfully
Dev Dataframe successfully reconstructed and aligned successfully


In [ ]:
model = LogisticRegression(max_iter=1000)
slot_model = LogisticRegression(max_iter=1000)

model.fit(X_train, Y_train)
print("Model Training Completed")
slot_model.fit(X_train_slots, Y_train_slots)
print("Slot Model Training Completed")